# ViTPose — From-Scratch Implementation on MS COCO

**Architecture:** Plain ViT-B encoder + Classic pose decoder  
**Reference:** Xu et al., *ViTPose: Simple Vision Transformer Baselines for Human Pose Estimation*, 2022  
**GPU:** RTX A4000 (16 GB)  
**Dataset:** MS COCO Keypoints 2017

In [1]:
# ============================================================
# CELL 1 — SYSTEM VERIFICATION
# ============================================================
import sys, torch

print('=' * 65)
print('ViTPose Implementation — System Verification')
print('=' * 65)
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA ver: {torch.version.cuda}')
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f'VRAM    : {vram/1e9:.1f} GB')
    DEVICE = torch.device('cuda:0')
else:
    print('WARNING: No GPU — training will be very slow')
    DEVICE = torch.device('cpu')

print(f'Device  : {DEVICE}')
print('=' * 65)

ViTPose Implementation — System Verification
Python  : 3.13.5
PyTorch : 2.7.1+cu118
CUDA    : True
CUDA ver: 11.8
GPU     : NVIDIA RTX A4000
VRAM    : 17.2 GB
Device  : cuda:0


In [2]:
# ============================================================
# CELL 2 — INSTALL DEPENDENCIES  (run once)
# ============================================================
import subprocess, sys

pkgs = ['pycocotools', 'timm', 'einops', 'tqdm', 'opencv-python']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=True)
    print(f'  [OK] {p}')

print('\nAll dependencies ready.')

  [OK] pycocotools
  [OK] timm
  [OK] einops
  [OK] tqdm
  [OK] opencv-python

All dependencies ready.


In [3]:
# ============================================================
# CELL 3 — IMPORTS
# ============================================================
import os, json, copy, time, math, random, warnings
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

import timm
from tqdm import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

print('Imports complete.')

Imports complete.


In [4]:
# ============================================================
# CELL 4 — CONFIGURATION
# ============================================================

# ─── COCO Dataset Paths ────────────────────────────────────
# Adjust COCO_ROOT to wherever the dataset is on the professor's machine
COCO_ROOT      = 'F:\MS COCO Dataset'  
TRAIN_IMG_DIR  = os.path.join(COCO_ROOT, 'train2017')
VAL_IMG_DIR    = os.path.join(COCO_ROOT, 'val2017')
TRAIN_ANN      = os.path.join(COCO_ROOT, 'annotations', 'person_keypoints_train2017.json')
VAL_ANN        = os.path.join(COCO_ROOT, 'annotations', 'person_keypoints_val2017.json')

# Verify
print('Checking dataset paths:')
all_ok = True
for p in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_ANN, VAL_ANN]:
    exists = os.path.exists(p)
    print(f'  {"[OK]" if exists else "[MISSING]"}  {p}')
    if not exists: all_ok = False

if not all_ok:
    print('\nWARNING: Some paths missing. Update COCO_ROOT above.')

# ─── Model Architecture (ViT-B — as in paper) ─────────────
PATCH_SIZE  = 16    # 16×16 pixel patches
IMG_H       = 256   # input height
IMG_W       = 192   # input width
HM_H        = 64    # heatmap height  (IMG_H / 4)
HM_W        = 48    # heatmap width   (IMG_W / 4)
NUM_JOINTS  = 17    # COCO-17 keypoints
D_MODEL     = 768   # ViT-B hidden dim
NUM_HEADS   = 12    # attention heads
NUM_LAYERS  = 12    # transformer depth
MLP_RATIO   = 4     # FFN expansion (hidden = 768 × 4 = 3072)
DECODE_CH   = 256   # decoder intermediate channels
SIGMA       = 2.0   # Gaussian sigma for heatmap GT (heatmap pixels)

# ─── Training Hyperparameters ─────────────────────────────
BATCH_SIZE    = 32    # RTX A4000 16 GB: safe at 32, reduce to 16 if OOM
LR            = 5e-4
WEIGHT_DECAY  = 0.1
NUM_EPOCHS    = 210   # full training schedule from paper
WARMUP_EPOCHS = 20
CKPT_EVERY    = 10    # save checkpoint every N epochs
USE_PRETRAIN  = True  # load MAE pretrained ViT-B (strongly recommended)
USE_AMP       = True  # mixed precision — saves ~30% VRAM
NUM_WORKERS   = 4
OUTPUT_DIR    = 'vitpose_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'checkpoints'), exist_ok=True)

print(f'\nConfig summary:')
print(f'  Model    : ViT-B (d={D_MODEL}, heads={NUM_HEADS}, layers={NUM_LAYERS})')
print(f'  Input    : {IMG_H}×{IMG_W}  →  Heatmap: {HM_H}×{HM_W}')
print(f'  Batch    : {BATCH_SIZE}   Epochs: {NUM_EPOCHS}   LR: {LR}')
print(f'  AMP      : {USE_AMP}   Pretrain: {USE_PRETRAIN}')

Checking dataset paths:
  [OK]  F:\MS COCO Dataset\train2017
  [OK]  F:\MS COCO Dataset\val2017
  [OK]  F:\MS COCO Dataset\annotations\person_keypoints_train2017.json
  [OK]  F:\MS COCO Dataset\annotations\person_keypoints_val2017.json

Config summary:
  Model    : ViT-B (d=768, heads=12, layers=12)
  Input    : 256×192  →  Heatmap: 64×48
  Batch    : 32   Epochs: 210   LR: 0.0005
  AMP      : True   Pretrain: True


In [6]:
# ============================================================
# CELL 5 — COCO DATASET LOADER  (CORRECTED)
# ============================================================

FLIP_PAIRS = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
JOINT_NAMES = [
    'nose','left_eye','right_eye','left_ear','right_ear',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_hip','right_hip',
    'left_knee','right_knee','left_ankle','right_ankle'
]
SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
IMG_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMG_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ─────────────────────────────────────────────────────────────
# Affine transform helpers  ← FIXED (was passing src[:2] = 2 rows,
#                             OpenCV needs all 3 rows)
# ─────────────────────────────────────────────────────────────

def _get_3rd_point(a, b):
    """
    Given two 2-D points a and b, return a third point c such that
    (a, b, c) forms a non-degenerate basis for computing the affine transform.
    c is perpendicular to the a→b segment.
    """
    d = a - b
    return b + np.array([-d[1], d[0]], dtype=np.float32)


def _rotate_dir(vec, rot_rad):
    """Rotate a 2-D vector by rot_rad (CCW)."""
    cs, sn = math.cos(rot_rad), math.sin(rot_rad)
    return np.array([vec[0]*cs - vec[1]*sn,
                     vec[0]*sn + vec[1]*cs], dtype=np.float32)


def get_affine_transform(center, scale, rot_deg, out_size):
    """
    Compute a 2×3 affine matrix that maps a (possibly rotated)
    rectangular crop region in the original image to a fixed
    output canvas of size (out_W, out_H).

    center  : np.array([cx, cy]) — center of the crop in orig image
    scale   : float              — width of the crop region (px)
    rot_deg : float              — rotation angle in degrees
    out_size: (W, H)             — output canvas dimensions

    Returns: (2, 3) float32 affine matrix for cv2.warpAffine
    """
    dst_w, dst_h = float(out_size[0]), float(out_size[1])
    rot_rad  = math.radians(rot_deg)

    # Height of source crop region (maintain output aspect ratio)
    src_h = scale * dst_h / dst_w

    # Direction from center toward the TOP of the source region (rotated)
    src_dir = _rotate_dir(np.array([0.0, -src_h * 0.5], dtype=np.float32), rot_rad)
    dst_dir = np.array([0.0, -dst_h * 0.5], dtype=np.float32)

    # ── 3 control points ──────────────────────────────────────
    src = np.zeros((3, 2), dtype=np.float32)
    dst = np.zeros((3, 2), dtype=np.float32)

    # Point 0 — crop center
    src[0] = center
    dst[0] = np.array([dst_w * 0.5, dst_h * 0.5], dtype=np.float32)

    # Point 1 — above center (in source direction, which may be rotated)
    src[1] = center + src_dir
    dst[1] = dst[0]  + dst_dir

    # Point 2 — perpendicular to 0→1  (makes the 3 points non-collinear)
    src[2] = _get_3rd_point(src[0], src[1])
    dst[2] = _get_3rd_point(dst[0], dst[1])

    # ── FIXED: pass full (3,2) arrays, NOT src[:2] ────────────
    trans = cv2.getAffineTransform(src, dst)   # src shape must be (3,2)
    return trans


def apply_affine(pt, trans):
    """Apply 2×3 affine matrix to a single (x, y) point."""
    return (trans @ np.array([pt[0], pt[1], 1.0], dtype=np.float32))[:2]


# ─────────────────────────────────────────────────────────────
# Gaussian heatmap generator
# ─────────────────────────────────────────────────────────────

def generate_heatmaps(keypoints, vis, sigma=SIGMA):
    """
    Generate 17 Gaussian heatmaps from keypoint coordinates.

    keypoints : (17, 2) in cropped IMG_W × IMG_H image space
    vis       : (17,)   COCO visibility flags (0/1/2)
    Returns:
      heatmaps : (17, HM_H, HM_W)  float32
      weights  : (17,)             1.0 if labeled, else 0.0
    """
    heatmaps = np.zeros((NUM_JOINTS, HM_H, HM_W), dtype=np.float32)
    weights  = np.zeros(NUM_JOINTS, dtype=np.float32)
    tmp      = int(sigma * 3)

    scale_x = HM_W / IMG_W
    scale_y = HM_H / IMG_H

    for j in range(NUM_JOINTS):
        if vis[j] < 1:
            continue

        mu_x = keypoints[j, 0] * scale_x
        mu_y = keypoints[j, 1] * scale_y

        if not (0 <= mu_x < HM_W and 0 <= mu_y < HM_H):
            continue

        x0 = max(0,    int(mu_x - tmp))
        y0 = max(0,    int(mu_y - tmp))
        x1 = min(HM_W, int(mu_x + tmp) + 1)
        y1 = min(HM_H, int(mu_y + tmp) + 1)

        xs = np.arange(x0, x1, dtype=np.float32) - mu_x
        ys = np.arange(y0, y1, dtype=np.float32) - mu_y
        xg, yg = np.meshgrid(xs, ys)
        g = np.exp(-(xg**2 + yg**2) / (2 * sigma**2))

        heatmaps[j, y0:y1, x0:x1] = np.maximum(heatmaps[j, y0:y1, x0:x1], g)
        weights[j] = 1.0

    return heatmaps, weights


# ─────────────────────────────────────────────────────────────
# Dataset class
# ─────────────────────────────────────────────────────────────

class COCOPoseDataset(Dataset):
    def __init__(self, img_dir, ann_file, is_train=True):
        self.img_dir  = img_dir
        self.is_train = is_train

        print(f'Loading annotations: {ann_file}')
        self.coco = COCO(ann_file)

        ann_ids  = self.coco.getAnnIds(catIds=self.coco.getCatIds(catNms=['person']))
        all_anns = self.coco.loadAnns(ann_ids)

        self.instances = [
            a for a in all_anns
            if a.get('num_keypoints', 0) > 0
            and a.get('area', 0) > 100
            and not a.get('iscrowd', 0)
        ]
        print(f'  {len(self.instances):,} person instances '
              f'({"train" if is_train else "val"})')

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        ann    = self.instances[idx]
        img_id = ann['image_id']

        # Load image
        info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.img_dir, info['file_name'])
        img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

        # Bbox → center, scale
        x, y, w, h = ann['bbox']
        cx    = x + w / 2.0
        cy    = y + h / 2.0
        scale = max(w, h * IMG_W / IMG_H) * 1.25   # 25 % padding

        # Raw keypoints
        kps_raw   = np.array(ann['keypoints'], dtype=np.float32).reshape(17, 3)
        keypoints = kps_raw[:, :2].copy()
        vis       = kps_raw[:, 2].copy()

        # Augmentation (training only)
        rot     = 0.0
        do_flip = False
        if self.is_train:
            rot     = (random.random() * 2 - 1) * 40.0
            scale  *= 0.7 + random.random() * 0.6
            cx     += scale * (random.random() - 0.5) * 0.1
            cy     += scale * (random.random() - 0.5) * 0.1
            do_flip = random.random() < 0.5

        # Affine crop  ← uses corrected get_affine_transform
        trans = get_affine_transform(
            np.array([cx, cy], dtype=np.float32),
            scale, rot, (IMG_W, IMG_H)
        )
        crop = cv2.warpAffine(img, trans, (IMG_W, IMG_H),
                               flags=cv2.INTER_LINEAR)

        # Transform keypoints to crop space
        for j in range(NUM_JOINTS):
            if vis[j] > 0:
                keypoints[j] = apply_affine(keypoints[j], trans)

        # Horizontal flip
        if do_flip:
            crop = crop[:, ::-1, :].copy()
            keypoints[:, 0] = IMG_W - 1 - keypoints[:, 0]
            for a, b in FLIP_PAIRS:
                keypoints[[a, b]] = keypoints[[b, a]]
                vis[[a, b]]       = vis[[b, a]]

        # Normalise
        crop_f = (crop.astype(np.float32) / 255.0 - IMG_MEAN) / IMG_STD
        img_t  = torch.from_numpy(crop_f.transpose(2, 0, 1))

        # Heatmaps
        hms, wts = generate_heatmaps(keypoints, vis)

        return {
            'image'    : img_t,
            'heatmaps' : torch.from_numpy(hms),
            'weights'  : torch.from_numpy(wts),
            'center'   : np.array([cx, cy], dtype=np.float32),
            'scale'    : np.float32(scale),
            'image_id' : img_id,
            'ann_id'   : ann['id'],
        }


# ── Sanity check ───────────────────────────────────────────────
print('\nBuilding val dataset (takes ~30 s)...')
val_dataset = COCOPoseDataset(VAL_IMG_DIR, VAL_ANN, is_train=False)
sample = val_dataset[0]
print(f'  image shape    : {sample["image"].shape}   (expected: [3, 256, 192])')
print(f'  heatmaps shape : {sample["heatmaps"].shape} (expected: [17, 64, 48])')
print(f'  visible joints : {int(sample["weights"].sum())}')
assert sample['image'].shape    == torch.Size([3, 256, 192]), 'image shape wrong!'
assert sample['heatmaps'].shape == torch.Size([17, 64, 48]), 'heatmap shape wrong!'
print('[OK] Dataset verified.')


Building val dataset (takes ~30 s)...
Loading annotations: F:\MS COCO Dataset\annotations\person_keypoints_val2017.json
loading annotations into memory...
Done (t=0.29s)
creating index...
index created!
  6,352 person instances (val)
  image shape    : torch.Size([3, 256, 192])   (expected: [3, 256, 192])
  heatmaps shape : torch.Size([17, 64, 48]) (expected: [17, 64, 48])
  visible joints : 10
[OK] Dataset verified.


In [7]:
# ============================================================
# CELL 6 — MODEL ARCHITECTURE (Implemented From Scratch)
# ============================================================
#
# Architecture (paper Fig. 2):
#
#  Input 256×192×3
#       │
#  PatchEmbedding (16×16 patches → 192 tokens × 768-d)
#       │
#  TransformerBlock × 12  ← Pre-LN, MHSA + FFN
#       │
#  Reshape to (B, 768, 16, 12)
#       │
#  ClassicDecoder
#    TransposedConv 768→256 ×2  →  (B, 256, 32, 24)
#    TransposedConv 256→256 ×2  →  (B, 256, 64, 48)
#    Conv1×1 256→17             →  (B, 17,  64, 48)
#       │
#  17 Heatmaps  ──argmax──►  17 (x,y) keypoints


# ─────────────────────────────────────────────────────────────
# A. PATCH EMBEDDING
# ─────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    """
    Split image into non-overlapping patches and project each
    patch into a d_model-dimensional vector.

    For 256×192 input, patch_size=16:
      Patches per row  : 192 / 16 = 12
      Patches per col  : 256 / 16 = 16
      Total tokens (N) : 12 × 16  = 192

    Implemented as Conv2d(stride=patch_size), equivalent to
    the linear projection in the ViT paper.
    """
    def __init__(self, img_h, img_w, patch_size, d_model):
        super().__init__()
        self.num_h = img_h // patch_size   # 16
        self.num_w = img_w // patch_size   # 12
        self.N     = self.num_h * self.num_w  # 192

        # Linear projection implemented as stride-p Conv2d
        self.proj = nn.Conv2d(3, d_model,
                               kernel_size=patch_size,
                               stride=patch_size, bias=True)

        # Learnable positional embedding: one vector per patch
        self.pos_embed = nn.Parameter(torch.zeros(1, self.N, d_model))
        self._init_pos_sincos(d_model)

        nn.init.trunc_normal_(self.proj.weight, std=0.02)
        nn.init.zeros_(self.proj.bias)

    def _init_pos_sincos(self, d_model):
        """2D sin-cos positional encoding (Dosovitskiy et al., 2021)."""
        assert d_model % 4 == 0
        gh = np.arange(self.num_h, dtype=np.float32)
        gw = np.arange(self.num_w, dtype=np.float32)
        gh, gw = np.meshgrid(gh, gw, indexing='ij')
        omega  = 1.0 / (10000 ** (np.arange(d_model//4, dtype=np.float32) / (d_model//4)))
        h_emb  = np.outer(gh.flatten(), omega)  # (N, d/4)
        w_emb  = np.outer(gw.flatten(), omega)
        pos    = np.concatenate([np.sin(h_emb), np.cos(h_emb),
                                  np.sin(w_emb), np.cos(w_emb)], axis=1)  # (N, d)
        with torch.no_grad():
            self.pos_embed.data.copy_(torch.from_numpy(pos).unsqueeze(0).float())

    def forward(self, x):
        """x: (B,3,H,W) → (B,N,d_model)"""
        x = self.proj(x)                         # (B, d, num_h, num_w)
        x = x.flatten(2).transpose(1, 2)          # (B, N, d)
        return x + self.pos_embed


# ─────────────────────────────────────────────────────────────
# B. MULTI-HEAD SELF-ATTENTION
# ─────────────────────────────────────────────────────────────
class MHSA(nn.Module):
    """
    Multi-Head Self-Attention.

    Paper: MHSA(F) = Concat(head_1,...,head_h) W^O
           head_i  = Attention(FW^Q_i, FW^K_i, FW^V_i)
           Attention(Q,K,V) = softmax(QK^T / √d_k) V

    Uses fused QKV projection for efficiency.
    For ViT-B: d_model=768, num_heads=12 → head_dim=64.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.h    = num_heads
        self.d_k  = d_model // num_heads   # 64
        self.scale= self.d_k ** -0.5       # 1/√64

        self.qkv      = nn.Linear(d_model, d_model * 3, bias=True)
        self.out_proj = nn.Linear(d_model, d_model,     bias=True)

        nn.init.trunc_normal_(self.qkv.weight,      std=0.02)
        nn.init.trunc_normal_(self.out_proj.weight,  std=0.02)
        nn.init.zeros_(self.qkv.bias)
        nn.init.zeros_(self.out_proj.bias)

    def forward(self, x):
        """x: (B,N,d) → (B,N,d)"""
        B, N, C = x.shape

        # QKV: (B, N, 3, h, d_k) → 3 × (B, h, N, d_k)
        qkv = self.qkv(x).reshape(B, N, 3, self.h, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)   # each (B, h, N, d_k)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, h, N, N)
        attn = F.softmax(attn, dim=-1)

        # Aggregate values: (B, h, N, d_k) → (B, N, d)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(x)


# ─────────────────────────────────────────────────────────────
# C. FEED-FORWARD NETWORK
# ─────────────────────────────────────────────────────────────
class FFN(nn.Module):
    """
    Two-layer MLP with GELU activation.

    Paper: FFN(x) = W₂ · GELU(W₁x + b₁) + b₂

    Hidden dim = d_model × mlp_ratio = 768 × 4 = 3072  (ViT-B)
    """
    def __init__(self, d_model, mlp_ratio=4):
        super().__init__()
        h = int(d_model * mlp_ratio)
        self.fc1 = nn.Linear(d_model, h, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(h, d_model, bias=True)
        nn.init.trunc_normal_(self.fc1.weight, std=0.02)
        nn.init.trunc_normal_(self.fc2.weight, std=0.02)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


# ─────────────────────────────────────────────────────────────
# D. TRANSFORMER BLOCK
# ─────────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    """
    One ViT Transformer block (Pre-LN variant).

    Paper equations:
      F'_{i+1} = F_i  + MHSA( LN(F_i)  )    ← attention sub-layer
      F_{i+1}  = F'_{i+1} + FFN( LN(F'_{i+1}) )  ← FFN sub-layer

    LayerNorm is applied BEFORE each sub-layer (Pre-LN),
    which is standard in modern ViT implementations.
    """
    def __init__(self, d_model, num_heads, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.attn  = MHSA(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.mlp   = FFN(d_model, mlp_ratio)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # attention + residual
        x = x + self.mlp (self.norm2(x))   # FFN       + residual
        return x


# ─────────────────────────────────────────────────────────────
# E. VITPOSE ENCODER  (Backbone)
# ─────────────────────────────────────────────────────────────
class ViTPoseEncoder(nn.Module):
    """
    ViTPose backbone = Patch Embedding + L Transformer Blocks.

    Outputs a 2D spatial feature map (B, d_model, num_h, num_w)
    ready for the pose decoder.
    """
    def __init__(self, img_h, img_w, patch_size, d_model,
                  num_heads, num_layers, mlp_ratio=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_h, img_w, patch_size, d_model)
        self.num_h = self.patch_embed.num_h
        self.num_w = self.patch_embed.num_w

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, mlp_ratio)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x):
        """x: (B,3,H,W) → (B, d_model, num_h, num_w)"""
        x = self.patch_embed(x)              # (B, N, d)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)                     # (B, N, d)
        B, N, C = x.shape
        return x.transpose(1,2).reshape(B, C, self.num_h, self.num_w)


# ─────────────────────────────────────────────────────────────
# F. CLASSIC POSE DECODER
# ─────────────────────────────────────────────────────────────
class ClassicDecoder(nn.Module):
    """
    Lightweight decoder (from ViTPose paper):
      TransposedConv 768→256 (×2 up) + BN + ReLU
      TransposedConv 256→256 (×2 up) + BN + ReLU
      Conv 1×1 256→17  →  17 keypoint heatmaps

    Input : (B, d_model, 16, 12)
    Output: (B, 17,      64, 48)
    """
    def __init__(self, d_model, decode_ch, num_joints):
        super().__init__()
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(d_model,   decode_ch, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(decode_ch), nn.ReLU(inplace=True)
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(decode_ch, decode_ch, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(decode_ch), nn.ReLU(inplace=True)
        )
        self.head = nn.Conv2d(decode_ch, num_joints, 1)

        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d)):
                nn.init.normal_(m.weight, std=0.001)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """x: (B,768,16,12) → (B,17,64,48)"""
        return self.head(self.up2(self.up1(x)))


# ─────────────────────────────────────────────────────────────
# G. FULL VITPOSE MODEL
# ─────────────────────────────────────────────────────────────
class ViTPose(nn.Module):
    """ViTPose = ViT Encoder + Classic Pose Decoder."""
    def __init__(self):
        super().__init__()
        self.encoder = ViTPoseEncoder(
            IMG_H, IMG_W, PATCH_SIZE, D_MODEL,
            NUM_HEADS, NUM_LAYERS, MLP_RATIO
        )
        self.decoder = ClassicDecoder(D_MODEL, DECODE_CH, NUM_JOINTS)

    def forward(self, x):
        """x: (B,3,256,192) → (B,17,64,48)"""
        return self.decoder(self.encoder(x))


# ── Shape verification ─────────────────────────────────────────
model = ViTPose()
_x  = torch.randn(2, 3, IMG_H, IMG_W)
_y  = model(_x)
assert _y.shape == (2, 17, HM_H, HM_W), f'Shape error: {_y.shape}'
print(f'Model output shape : {_y.shape}  (expected (2, 17, 64, 48))')

total_p = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total parameters   : {total_p:.1f} M')
print(f'  (ViT-B backbone ≈ 86 M + decoder ≈ 2 M = 88 M total expected)')
del _x, _y
print('[OK] Architecture verified.')

Model output shape : torch.Size([2, 17, 64, 48])  (expected (2, 17, 64, 48))
Total parameters   : 90.0 M
  (ViT-B backbone ≈ 86 M + decoder ≈ 2 M = 88 M total expected)
[OK] Architecture verified.


In [8]:
# ============================================================
# CELL 7 — LOAD MAE PRETRAINED WEIGHTS  
# ============================================================

import os, math, requests
import torch
import torch.nn.functional as F

# ── Install safetensors if not present ────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'safetensors', '-q'], check=True)
from safetensors.torch import load_file as _load_safetensors
print('[OK] safetensors ready')

# ── Download config ───────────────────────────────────────────
WEIGHTS_DIR   = OUTPUT_DIR          # reuse vitpose_output/
WEIGHTS_PATH  = os.path.join(WEIGHTS_DIR, 'vit_base_mae.safetensors')

# Direct HuggingFace CDN URL — no auth needed (public model)
HF_URL = (
    'https://huggingface.co/timm/vit_base_patch16_224.mae'
    '/resolve/main/model.safetensors'
)

# ── Download with plain requests (no tqdm, no xet) ───────────
def download_direct(url, save_path):
    """
    Download a file using requests.get with a simple text progress line.
    Completely avoids HuggingFace Hub / xet / tqdm-notebook.
    """
    print(f'Downloading ViT-B MAE weights ...')
    print(f'  URL  : {url}')
    print(f'  Save : {save_path}')

    resp = requests.get(url, stream=True, timeout=600, allow_redirects=True)
    resp.raise_for_status()

    total      = int(resp.headers.get('content-length', 0))
    downloaded = 0
    chunk_size = 4 * 1024 * 1024   # 4 MB chunks

    with open(save_path, 'wb') as fout:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            if chunk:
                fout.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded * 100 / total
                    print(
                        f'  {downloaded/1e6:>6.0f} / {total/1e6:.0f} MB'
                        f'  ({pct:5.1f}%)',
                        end='\r', flush=True
                    )
    print(f'\n  Download complete: {save_path}')


# ── Positional embedding interpolation ────────────────────────
def interpolate_pos_embed(timm_pos, tgt_h, tgt_w):
    """
    timm pos_embed : (1, 197, 768)  [CLS + 14×14 patches for 224×224]
    Our  pos_embed : (1, 192, 768)  [no CLS, 16×12 patches for 256×192]

    Bilinearly interpolates 14×14 → 16×12.
    """
    pe   = timm_pos[:, 1:, :]                               # drop CLS  (1,196,768)
    orig = int(math.sqrt(pe.shape[1]))                      # 14
    pe   = pe.reshape(1, orig, orig, -1).permute(0,3,1,2).float()   # (1,768,14,14)
    pe   = F.interpolate(pe, size=(tgt_h, tgt_w), mode='bicubic', align_corners=False)
    pe   = pe.permute(0,2,3,1).reshape(1, tgt_h*tgt_w, -1)         # (1,192,768)
    return pe


# ── Main weight loading ───────────────────────────────────────
if USE_PRETRAIN:
    try:
        # Download only once; reuse cached file on subsequent runs
        if os.path.exists(WEIGHTS_PATH):
            print(f'Using cached weights: {WEIGHTS_PATH}')
        else:
            download_direct(HF_URL, WEIGHTS_PATH)

        # Load safetensors → plain dict of {name: tensor}
        print('Loading state dict from file ...')
        ts = _load_safetensors(WEIGHTS_PATH, device='cpu')
        print(f'  Keys in file: {len(ts)}')

        ms = model.state_dict()

        # ── timm key  →  our model key ────────────────────────
        mapping = {
            'patch_embed.proj.weight' : 'encoder.patch_embed.proj.weight',
            'patch_embed.proj.bias'   : 'encoder.patch_embed.proj.bias',
            'norm.weight'             : 'encoder.norm.weight',
            'norm.bias'               : 'encoder.norm.bias',
        }
        for i in range(NUM_LAYERS):
            t = f'blocks.{i}'
            o = f'encoder.blocks.{i}'
            mapping.update({
                f'{t}.norm1.weight'    : f'{o}.norm1.weight',
                f'{t}.norm1.bias'      : f'{o}.norm1.bias',
                f'{t}.attn.qkv.weight' : f'{o}.attn.qkv.weight',
                f'{t}.attn.qkv.bias'   : f'{o}.attn.qkv.bias',
                f'{t}.attn.proj.weight': f'{o}.attn.out_proj.weight',
                f'{t}.attn.proj.bias'  : f'{o}.attn.out_proj.bias',
                f'{t}.norm2.weight'    : f'{o}.norm2.weight',
                f'{t}.norm2.bias'      : f'{o}.norm2.bias',
                f'{t}.mlp.fc1.weight'  : f'{o}.mlp.fc1.weight',
                f'{t}.mlp.fc1.bias'    : f'{o}.mlp.fc1.bias',
                f'{t}.mlp.fc2.weight'  : f'{o}.mlp.fc2.weight',
                f'{t}.mlp.fc2.bias'    : f'{o}.mlp.fc2.bias',
            })

        # Copy matched weights
        loaded, missing = 0, []
        for tk, ok in mapping.items():
            if tk in ts and ok in ms:
                ms[ok].copy_(ts[tk].to(ms[ok].dtype))
                loaded += 1
            else:
                missing.append(tk)

        # Positional embedding — interpolate 14×14 → 16×12
        if 'pos_embed' in ts:
            pe = interpolate_pos_embed(
                ts['pos_embed'],
                model.encoder.patch_embed.num_h,   # 16
                model.encoder.patch_embed.num_w     # 12
            )
            ms['encoder.patch_embed.pos_embed'].copy_(pe)
            print('  pos_embed : interpolated 14×14 → 16×12  [OK]')

        model.load_state_dict(ms)
        del ts

        print(f'  Weights loaded : {loaded} / {len(mapping)}')
        if missing:
            print(f'  Missing keys   : {missing}')
        print('  Decoder head   : randomly initialised (trained on COCO)')
        print('[OK] Pretrained weights loaded.')

    except requests.exceptions.ConnectionError:
        print('[WARNING] No internet connection.')
        print('  Proceeding with random initialisation.')
        print('  Training still works but will take more epochs to converge.')

    except Exception as e:
        print(f'[WARNING] Weight loading failed: {type(e).__name__}: {e}')
        print('  Proceeding with random initialisation.')

else:
    print('USE_PRETRAIN = False — random initialisation.')

model = model.to(DEVICE)
print(f'\nModel on: {DEVICE}')

[OK] safetensors ready
Using cached weights: vitpose_output\vit_base_mae.safetensors
Loading state dict from file ...
  Keys in file: 150
  pos_embed : interpolated 14×14 → 16×12  [OK]
  Weights loaded : 148 / 148
  Decoder head   : randomly initialised (trained on COCO)
[OK] Pretrained weights loaded.

Model on: cuda:0


In [9]:
# ============================================================
# CELL 8 — TRAINING SETUP
# ============================================================

# ─── Loss Function ────────────────────────────────────────────
class WeightedMSE(nn.Module):
    """
    MSE loss on heatmaps, weighted by joint visibility.
    Invisible joints (weight=0) do not contribute.
    """
    def forward(self, pred, target, weight):
        # weight: (B,17) → (B,17,1,1) for broadcasting
        w = weight[:, :, None, None].expand_as(pred)
        return ((pred - target)**2 * w).sum() / (w.sum() + 1e-6)

criterion = WeightedMSE()

# ─── DataLoaders ─────────────────────────────────────────────
print('Building train dataset (takes ~2 min)...')
train_dataset = COCOPoseDataset(TRAIN_IMG_DIR, TRAIN_ANN, is_train=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset):,} instances → {len(train_loader)} batches')
print(f'Val:   {len(val_dataset):,}   instances → {len(val_loader)} batches')

# ─── Optimizer (AdamW, separate decay/no-decay groups) ───────
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    (no_decay if ('bias' in name or 'norm' in name) else decay).append(p)

optimizer = AdamW(
    [{'params': decay,    'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=LR
)

# ─── Cosine LR with linear warmup ────────────────────────────
def lr_lambda(ep):
    if ep < WARMUP_EPOCHS:
        return max(ep / WARMUP_EPOCHS, 1e-6)
    prog = (ep - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * prog))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = GradScaler(enabled=USE_AMP)

print(f'\nOptimizer : AdamW  lr={LR}  wd={WEIGHT_DECAY}')
print(f'Scheduler : Cosine with {WARMUP_EPOCHS}-epoch warmup')
print('[OK] Training setup complete.')

Building train dataset (takes ~2 min)...
Loading annotations: F:\MS COCO Dataset\annotations\person_keypoints_train2017.json
loading annotations into memory...
Done (t=6.49s)
creating index...
index created!
  149,813 person instances (train)
Train: 149,813 instances → 4681 batches
Val:   6,352   instances → 199 batches

Optimizer : AdamW  lr=0.0005  wd=0.1
Scheduler : Cosine with 20-epoch warmup
[OK] Training setup complete.


In [10]:
# ============================================================
# DIAGNOSTIC — Find why images can't be read
# ============================================================
import os

print(f'COCO_ROOT     : {COCO_ROOT}')
print(f'TRAIN_IMG_DIR : {TRAIN_IMG_DIR}')
print(f'VAL_IMG_DIR   : {VAL_IMG_DIR}')
print()

# Check if dirs exist
for d in [TRAIN_IMG_DIR, VAL_IMG_DIR]:
    exists = os.path.isdir(d)
    if exists:
        files = os.listdir(d)
        print(f'[OK] {d}')
        print(f'     Contains {len(files)} files')
        if files:
            print(f'     First file: {files[0]}')
        else:
            print(f'     *** EMPTY DIRECTORY ***')
    else:
        print(f'[MISSING] {d}')

print()

# Show what IS in COCO_ROOT
print(f'Contents of {COCO_ROOT}:')
for item in sorted(os.listdir(COCO_ROOT)):
    full = os.path.join(COCO_ROOT, item)
    size = os.path.getsize(full) if os.path.isfile(full) else '-'
    kind = 'DIR' if os.path.isdir(full) else 'FILE'
    print(f'  {kind:4}  {item}   {size if isinstance(size,str) else f"{size/1e6:.0f} MB"}')

print()

# Check a specific image path
ann  = val_dataset.instances[0]
info = val_dataset.coco.loadImgs(ann['image_id'])[0]
path = os.path.join(VAL_IMG_DIR, info['file_name'])
print(f'Sample image path : {path}')
print(f'File exists       : {os.path.exists(path)}')

COCO_ROOT     : F:\MS COCO Dataset
TRAIN_IMG_DIR : F:\MS COCO Dataset\train2017
VAL_IMG_DIR   : F:\MS COCO Dataset\val2017

[OK] F:\MS COCO Dataset\train2017
     Contains 10785 files
     First file: 000000000009.jpg
[OK] F:\MS COCO Dataset\val2017
     Contains 5000 files
     First file: 000000000139.jpg

Contents of F:\MS COCO Dataset:
  DIR   annotations   -
  FILE  annotations_trainval2017.zip   253 MB
  FILE  desktop.ini   0 MB
  DIR   train2017   -
  FILE  train2017.zip   19337 MB
  DIR   val2017   -
  FILE  val2017.zip   816 MB

Sample image path : F:\MS COCO Dataset\val2017\000000425226.jpg
File exists       : True


In [11]:
# ============================================================
# FIX — Filter training dataset to only images on disk
# ============================================================
import os, time

print('Scanning train2017 folder...', flush=True)
# Get all filenames that actually exist (fast set lookup)
existing_files = set(os.listdir(TRAIN_IMG_DIR))
print(f'  Images on disk   : {len(existing_files):,}')
print(f'  Annotations total: {len(train_dataset.instances):,}')

t0 = time.time()
valid_instances = []
for ann in train_dataset.instances:
    info = train_dataset.coco.loadImgs(ann['image_id'])[0]
    if info['file_name'] in existing_files:
        valid_instances.append(ann)

train_dataset.instances = valid_instances
print(f'  Valid instances  : {len(valid_instances):,}  '
      f'(filtered in {time.time()-t0:.1f}s)', flush=True)

# Rebuild DataLoader with filtered dataset
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 0,
    pin_memory  = False,
    drop_last   = True,
)
print(f'  Batches per epoch: {len(train_loader)}', flush=True)

Scanning train2017 folder...
  Images on disk   : 10,785
  Annotations total: 149,813
  Valid instances  : 13,686  (filtered in 0.2s)
  Batches per epoch: 427


In [12]:
print('Testing DataLoader...', flush=True)
import time
t0 = time.time()
test_batch = next(iter(train_loader))
print(f'First batch OK  shape={test_batch["image"].shape}  '
      f'time={time.time()-t0:.1f}s', flush=True)
print('[OK] Ready to train. Run Cell 9 now.', flush=True)

Testing DataLoader...
First batch OK  shape=torch.Size([32, 3, 256, 192])  time=0.5s
[OK] Ready to train. Run Cell 9 now.


In [13]:
# ============================================================
# WINDOWS DATALOADER FIX — Run this before Cell 9
# ============================================================
# On Windows, DataLoader with num_workers > 0 inside Jupyter
# causes workers to freeze permanently (known Windows/multiprocessing bug).
# Setting num_workers = 0 fixes it — data loads in the main thread.

import torch, os
from torch.utils.data import DataLoader

# Rebuild DataLoaders with num_workers = 0
train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle    = True,
    num_workers = 0,      # <── CRITICAL FIX for Windows + Jupyter
    pin_memory  = False,  # must be False when num_workers=0 on some setups
    drop_last   = True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = 0,      # <── same fix
    pin_memory  = False,
)

# Verify with a single batch (should complete in <30 seconds)
print('Testing DataLoader... (should finish in <30 sec)', flush=True)
import time
t0 = time.time()
test_batch = next(iter(train_loader))
elapsed = time.time() - t0
print(f'First batch loaded in {elapsed:.1f}s  — shape: {test_batch["image"].shape}', flush=True)
print('[OK] DataLoader working. Now run Cell 9.', flush=True)

Testing DataLoader... (should finish in <30 sec)
First batch loaded in 0.4s  — shape: torch.Size([32, 3, 256, 192])
[OK] DataLoader working. Now run Cell 9.


In [14]:
# ============================================================
# CELL 9 — TRAINING LOOP  (FIXED — no tqdm dependency)
# ============================================================

import sys, time, math

# Re-enable tqdm in case Cell 7 disabled it
os.environ.pop("TQDM_DISABLE", None)
os.environ.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)

# ── Resume config ─────────────────────────────────────────────
# Set to a checkpoint path to resume, e.g.:
#   RESUME_FROM = 'vitpose_output/checkpoints/epoch_009.pth'
# RESUME_FROM = None
RESUME_FROM = 'vitpose_output/checkpoints/epoch_109.pth'

start_epoch  = 0
best_loss    = float('inf')
train_losses = []

if RESUME_FROM and os.path.exists(RESUME_FROM):
    print(f'Loading checkpoint: {RESUME_FROM}')
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch  = ckpt['epoch'] + 1
    train_losses = ckpt.get('train_losses', [])
    print(f'  Resumed at epoch {start_epoch}', flush=True)
else:
    print('Starting from epoch 0 (no checkpoint set)', flush=True)

if start_epoch >= NUM_EPOCHS:
    print(f'Already finished! start_epoch={start_epoch} >= NUM_EPOCHS={NUM_EPOCHS}')
    print('To retrain, set start_epoch=0 or delete old checkpoints.')
else:
    print(f'Training: epoch {start_epoch} → {NUM_EPOCHS}')
    print(f'Checkpoint every {CKPT_EVERY} epochs → {OUTPUT_DIR}/checkpoints/')
    print('─' * 65, flush=True)

    def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch):
        """Train for one epoch. Returns average loss."""
        model.train()
        total_loss  = 0.0
        num_batches = len(loader)
        report_every = max(1, num_batches // 10)  # print ~10 times per epoch

        for batch_idx, batch in enumerate(loader):
            imgs = batch['image'].to(DEVICE, non_blocking=True)
            hms  = batch['heatmaps'].to(DEVICE, non_blocking=True)
            wts  = batch['weights'].to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with autocast(enabled=USE_AMP):
                pred = model(imgs)
                loss = criterion(pred, hms, wts)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            # Print batch progress periodically
            if (batch_idx + 1) % report_every == 0:
                avg_so_far = total_loss / (batch_idx + 1)
                print(
                    f'  Epoch {epoch:03d}  batch {batch_idx+1:4d}/{num_batches}'
                    f'  loss={loss.item():.4f}  avg={avg_so_far:.4f}',
                    flush=True
                )

        return total_loss / num_batches


    # ── Main loop ─────────────────────────────────────────────
    for epoch in range(start_epoch, NUM_EPOCHS):
        t0 = time.time()

        epoch_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, epoch
        )
        train_losses.append(epoch_loss)
        scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']
        elapsed    = time.time() - t0

        print(
            f'Epoch {epoch:03d}/{NUM_EPOCHS-1}  '
            f'loss={epoch_loss:.4f}  '
            f'lr={current_lr:.2e}  '
            f'time={elapsed:.0f}s',
            flush=True
        )

        # Save checkpoint every CKPT_EVERY epochs
        if (epoch + 1) % CKPT_EVERY == 0 or epoch == NUM_EPOCHS - 1:
            ckpt_path = os.path.join(
                OUTPUT_DIR, 'checkpoints', f'epoch_{epoch:03d}.pth'
            )
            torch.save({
                'epoch'          : epoch,
                'model_state'    : model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'loss'           : epoch_loss,
                'train_losses'   : train_losses,
            }, ckpt_path)
            print(f'  --> Checkpoint saved: {ckpt_path}', flush=True)

    print('\nTraining complete!', flush=True)

Loading checkpoint: vitpose_output/checkpoints/epoch_109.pth
  Resumed at epoch 110
Training: epoch 110 → 210
Checkpoint every 10 epochs → vitpose_output/checkpoints/
─────────────────────────────────────────────────────────────────
  Epoch 110  batch   42/427  loss=0.0005  avg=0.0006
  Epoch 110  batch   84/427  loss=0.0006  avg=0.0006
  Epoch 110  batch  126/427  loss=0.0007  avg=0.0006
  Epoch 110  batch  168/427  loss=0.0006  avg=0.0006
  Epoch 110  batch  210/427  loss=0.0005  avg=0.0006
  Epoch 110  batch  252/427  loss=0.0006  avg=0.0006
  Epoch 110  batch  294/427  loss=0.0005  avg=0.0006
  Epoch 110  batch  336/427  loss=0.0006  avg=0.0006
  Epoch 110  batch  378/427  loss=0.0006  avg=0.0006
  Epoch 110  batch  420/427  loss=0.0006  avg=0.0006
Epoch 110/209  loss=0.0006  lr=2.67e-04  time=212s
  Epoch 111  batch   42/427  loss=0.0006  avg=0.0006
  Epoch 111  batch   84/427  loss=0.0004  avg=0.0005
  Epoch 111  batch  126/427  loss=0.0005  avg=0.0005
  Epoch 111  batch  168/427

In [14]:
# ============================================================
# CELL 10 — COCO EVALUATION  (AP, AP50, AP75, APM, APL, AR)
# ============================================================

# Optional: load a specific checkpoint for evaluation
EVAL_CKPT = 'vitpose_output/checkpoints/epoch_209.pth'
if EVAL_CKPT and os.path.exists(EVAL_CKPT):
    ckpt = torch.load(EVAL_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    print(f'Loaded checkpoint: {EVAL_CKPT} (epoch {ckpt["epoch"]})')


def decode_heatmaps(hms, center, scale):
    """
    Convert 17 predicted heatmaps → 17 (x, y, score) keypoints
    in original image coordinates.

    hms    : (17, HM_H, HM_W)  numpy
    center : (2,)  (cx, cy) of the crop in original image
    scale  : float

    Returns: (17, 3) — columns: x, y, score  (original image space)
    """
    kps  = np.zeros((NUM_JOINTS, 3), dtype=np.float32)
    # Inverse affine: crop → original image
    trans     = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    inv_trans = cv2.invertAffineTransform(trans)

    for j in range(NUM_JOINTS):
        hm = hms[j]                        # (HM_H, HM_W)
        flat = int(np.argmax(hm))
        iy, ix = divmod(flat, HM_W)
        score  = float(hm[iy, ix])

        # ±0.25 sub-pixel refinement (ViTPose paper post-processing)
        dy = 0.25 * np.sign(hm[iy+1,ix] - hm[iy-1,ix]) if 0<iy<HM_H-1 else 0.0
        dx = 0.25 * np.sign(hm[iy,ix+1] - hm[iy,ix-1]) if 0<ix<HM_W-1 else 0.0

        # Heatmap → crop image space
        px_crop = (ix + dx + 0.5) * (IMG_W / HM_W)
        py_crop = (iy + dy + 0.5) * (IMG_H / HM_H)

        # Crop → original image space
        pt = inv_trans @ np.array([px_crop, py_crop, 1.0])
        kps[j] = [pt[0], pt[1], score]

    return kps


def run_inference(model, loader):
    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Inference'):
            imgs    = batch['image'].to(DEVICE, non_blocking=True)
            centers = batch['center'].numpy()
            scales  = batch['scale'].numpy()
            img_ids = batch['image_id']

            with autocast(enabled=USE_AMP):
                pred_hms = model(imgs)     # (B, 17, 64, 48)

            pred_hms = pred_hms.cpu().numpy()

            for b in range(pred_hms.shape[0]):
                kps   = decode_heatmaps(pred_hms[b], centers[b], float(scales[b]))
                score = float(np.mean(sorted(kps[:,2], reverse=True)[:10]))
                predictions.append({
                    'image_id'   : int(img_ids[b]),
                    'category_id': 1,
                    'keypoints'  : kps.flatten().tolist(),  # 51 values
                    'score'      : score,
                })

    return predictions


def coco_eval(ann_file, predictions):
    pred_path = os.path.join(OUTPUT_DIR, 'predictions.json')
    with open(pred_path, 'w') as f:
        json.dump(predictions, f)

    coco_gt   = COCO(ann_file)
    coco_dt   = coco_gt.loadRes(pred_path)
    evaluator = COCOeval(coco_gt, coco_dt, iouType='keypoints')
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    s = evaluator.stats
    res = dict(AP=s[0], AP50=s[1], AP75=s[2], APM=s[3], APL=s[4], AR=s[5])

    print('\n' + '='*55)
    print('COCO KEYPOINT EVALUATION')
    print('='*55)
    print(f'  AP    (0.50:0.95) : {res["AP"]*100:6.1f}')
    print(f'  AP50  (0.50)      : {res["AP50"]*100:6.1f}')
    print(f'  AP75  (0.75)      : {res["AP75"]*100:6.1f}')
    print(f'  APM   (medium)    : {res["APM"]*100:6.1f}')
    print(f'  APL   (large)     : {res["APL"]*100:6.1f}')
    print(f'  AR    (0.50:0.95) : {res["AR"]*100:6.1f}')
    print('='*55)
    print(f'  Paper ViTPose-B   :   75.8   90.7   83.2   72.3   82.0   80.9')
    print('='*55)
    return res


# ── Run evaluation ────────────────────────────────────────────
print('Running inference on val2017...')
preds  = run_inference(model, val_loader)
print(f'Predictions: {len(preds):,}')

results = coco_eval(VAL_ANN, preds)

Loaded checkpoint: vitpose_output/checkpoints/epoch_209.pth (epoch 209)
Running inference on val2017...


Inference: 100%|█████████████████████████████████████████████████████████████████████| 199/199 [01:11<00:00,  2.80it/s]


Predictions: 6,352
loading annotations into memory...
Done (t=0.26s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.29s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=2.06s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.462
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.796
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.487
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.454
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.479
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.511
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.818
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.546
 Average Recall     (AR) @[ IoU=0.50:0.95

In [16]:
# ============================================================
# CELL 11 — VISUALIZATION
# ============================================================

BONE_COLORS = [
    (255,0,0),(255,128,0),(255,255,0),(128,255,0),(0,255,0),
    (0,255,128),(0,255,255),(0,128,255),(0,0,255),(128,0,255),
    (255,0,255),(255,0,128),(200,100,50),(50,200,100),(100,50,200),(150,150,0)
]

def draw_skeleton_on(img_rgb, kps, thresh=0.1):
    out = img_rgb.copy()
    for idx, (a, b) in enumerate(SKELETON):
        if kps[a,2] > thresh and kps[b,2] > thresh:
            cv2.line(out,
                      (int(kps[a,0]), int(kps[a,1])),
                      (int(kps[b,0]), int(kps[b,1])),
                      BONE_COLORS[idx % len(BONE_COLORS)], 2, cv2.LINE_AA)
    for j in range(NUM_JOINTS):
        if kps[j,2] > thresh:
            cv2.circle(out, (int(kps[j,0]),int(kps[j,1])), 4, (255,255,255), -1)
            cv2.circle(out, (int(kps[j,0]),int(kps[j,1])), 4, (0,0,220),     1)
    return out


# ── Plot training loss ────────────────────────────────────────
if train_losses:
    plt.figure(figsize=(10, 4))
    plt.plot(train_losses, color='steelblue', label='Train Loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
    plt.title('ViTPose Training Loss')
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=150)
    plt.show()


# ── Qualitative results (GT vs Prediction side-by-side) ───────
model.eval()
coco_gt = COCO(VAL_ANN)
N_SHOW  = 4

sample_ids = random.sample(coco_gt.getAnnIds(catIds=1), N_SHOW)
fig, axes  = plt.subplots(2, N_SHOW, figsize=(5*N_SHOW, 10))

for col, aid in enumerate(sample_ids):
    ann  = coco_gt.loadAnns([aid])[0]
    info = coco_gt.loadImgs(ann['image_id'])[0]
    img  = cv2.cvtColor(cv2.imread(os.path.join(VAL_IMG_DIR, info['file_name'])),
                         cv2.COLOR_BGR2RGB)

    # Bbox → crop
    x,y,w,h = ann['bbox']
    cx, cy = x+w/2, y+h/2
    scale  = max(w, h*IMG_W/IMG_H)*1.25
    trans  = get_affine_transform(np.array([cx,cy]), scale, 0., (IMG_W,IMG_H))
    crop   = cv2.warpAffine(img, trans, (IMG_W,IMG_H))

    # Normalise → infer
    inp = (crop.astype(np.float32)/255. - IMG_MEAN) / IMG_STD
    inp = torch.from_numpy(inp.transpose(2,0,1)).unsqueeze(0).to(DEVICE)

    with torch.no_grad(), autocast(enabled=USE_AMP):
        hm = model(inp)[0].cpu().numpy()

    pred_kps = decode_heatmaps(hm, np.array([cx,cy]), scale)  # (17,3) original

    # Ground truth in original image
    gt_raw = np.array(ann['keypoints']).reshape(17,3).astype(np.float32)
    gt_raw[:,2] = (gt_raw[:,2] > 0).astype(np.float32)

    # Tight crop region for display
    pad = 20
    r_x1 = max(0, int(cx-scale/2)-pad)
    r_y1 = max(0, int(cy-scale*IMG_H/IMG_W/2)-pad)
    r_x2 = min(img.shape[1], r_x1+int(scale)+2*pad)
    r_y2 = min(img.shape[0], r_y1+int(scale*IMG_H/IMG_W)+2*pad)

    axes[0,col].imshow(draw_skeleton_on(img, gt_raw,   thresh=0.5)[r_y1:r_y2, r_x1:r_x2])
    axes[0,col].set_title('Ground Truth', fontsize=11)
    axes[0,col].axis('off')

    axes[1,col].imshow(draw_skeleton_on(img, pred_kps, thresh=0.1)[r_y1:r_y2, r_x1:r_x2])
    axes[1,col].set_title('Predicted', fontsize=11)
    axes[1,col].axis('off')

plt.suptitle('ViTPose — Predictions vs Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR}/sample_predictions.png')

NameError: name 'train_losses' is not defined

In [ ]:
# ============================================================
# CELL 12 — EXPERIMENTS TABLE
# ============================================================
# After running experiments (changing decoder / input resolution /
# fine-tuning strategy), record results here and print the comparison
# table to show the professor.

# Add your results manually below:
EXP_RESULTS = {
    # Format: 'Experiment Name': {'AP': x, 'AP50': y, 'AP75': z, 'AR': w}
    # ─── Fill these after running each experiment ───
    'Baseline (ViTPose-B, 256×192, Classic Decoder)': {
        'AP': 0.0, 'AP50': 0.0, 'AP75': 0.0, 'AR': 0.0
    },
    # Experiment A — Simple Decoder (replace ClassicDecoder with bilinear upsample)
    # 'Exp A: Simple Decoder': {'AP':0., 'AP50':0., 'AP75':0., 'AR':0.},
    # Experiment B — Higher resolution 320×240
    # 'Exp B: 320×240 Input':  {'AP':0., 'AP50':0., 'AP75':0., 'AR':0.},
    # Experiment C — Freeze MHSA (only train FFN + decoder)
    # 'Exp C: Freeze MHSA':    {'AP':0., 'AP50':0., 'AP75':0., 'AR':0.},
}

# ── If just ran eval, add current result ─────────────────────
if 'results' in dir():
    EXP_RESULTS['Baseline (ViTPose-B, 256×192, Classic Decoder)'] = {
        'AP'  : results['AP'],
        'AP50': results['AP50'],
        'AP75': results['AP75'],
        'AR'  : results['AR'],
    }

# ── Print results table ───────────────────────────────────────
print('\n' + '='*75)
print('EXPERIMENT RESULTS vs. PAPER')
print('='*75)
print(f'{"Experiment":<42} {"AP":>6} {"AP50":>6} {"AP75":>6} {"AR":>6}')
print('─'*75)

for name, r in EXP_RESULTS.items():
    print(f'{name:<42} '
          f'{r["AP"]*100:>6.1f} '
          f'{r["AP50"]*100:>6.1f} '
          f'{r["AP75"]*100:>6.1f} '
          f'{r["AR"]*100:>6.1f}')

print('─'*75)
print(f'{"Paper: ViTPose-B (256×192, GT bbox)":<42} '
      f'{"75.8":>6} {"90.7":>6} {"83.2":>6} {"80.9":>6}')
print('='*75)

print('\nNote: Paper uses detected bboxes from a detector for testing.')
print('      Our eval uses COCO GT bboxes → results may differ.')